# Lab 02: Webhook Integration - Testing with Stripe CLI

**Lab**: 02-webhook-integration  
**Duration**: ~15 minutes  
**Prerequisites**: Completed `02_build_server.ipynb`, Stripe CLI installed

## Learning Objectives

By the end of this notebook, you will:
- Set up Stripe CLI for local testing
- Forward webhooks to your local server
- Trigger test events and verify handling

---

## Setup

Let's verify the Stripe CLI is installed.

<!-- PRESENTER: Help attendees install Stripe CLI if needed -->

In [1]:
# Verify Stripe CLI is installed
!stripe version

stripe version 1.32.0
A newer version of the Stripe CLI is available, please update to: v1.35.0


### Installing Stripe CLI

If not installed, follow the [installation guide](https://stripe.com/docs/stripe-cli#install):

```bash
# macOS (Homebrew)
brew install stripe/stripe-cli/stripe

# Windows (Scoop)
scoop install stripe

# Linux
# Download from: https://github.com/stripe/stripe-cli/releases
```

## Step 1: Login to Stripe CLI

The CLI needs to authenticate with your Stripe account.

**Run this in a terminal** (not in Jupyter):

```bash
stripe login
```

This opens a browser to authenticate. Follow the prompts.

In [3]:
# Check if logged in
!stripe config --list 2>/dev/null | head -5 || echo "Run 'stripe login' in a terminal first"

color = ''
installed_plugins = ['apps']
project-name = 'default'

[default]


## Step 2: The Three-Terminal Setup

Testing webhooks locally requires **3 terminals**:

```
+------------------+     +------------------+     +------------------+
|   Terminal 1     |     |   Terminal 2     |     |   Terminal 3     |
|                  |     |                  |     |                  |
|  Flask Server    |     |  stripe listen   |     |  stripe trigger  |
|  (receives)      | <-- |  (forwards)      | <-- |  (sends events)  |
+------------------+     +------------------+     +------------------+
```

<!-- PRESENTER: Open 3 terminal windows/tabs and arrange them -->

## Step 3: Start the Flask Server

**In Terminal 1**, start the webhook server:

```bash
cd 02-webhook-integration
source .venv/bin/activate  # or: uv sync && source .venv/bin/activate
flask --app src/server run --port=4242
```

You should see:
```
 * Running on http://127.0.0.1:4242
```

## Step 4: Start Stripe Listen

**In Terminal 2**, start forwarding webhooks:

```bash
stripe listen --forward-to localhost:4242/webhook
```

You should see:
```
> Ready! Your webhook signing secret is whsec_xxx (^C to quit)
```

**Important**: Copy the `whsec_xxx` value and update your `.env` file!

## Step 5: Trigger Test Events

**In Terminal 3**, trigger events:

```bash
stripe trigger payment_intent.succeeded
```

This creates a test PaymentIntent and triggers the `payment_intent.succeeded` event.

### What You Should See

**Terminal 2 (stripe listen)**:
```
2024-01-15 10:30:00 --> payment_intent.succeeded [evt_xxx]
2024-01-15 10:30:00 <-- [200] POST http://localhost:4242/webhook
```

**Terminal 1 (Flask server)**:
```
Payment for 2000 succeeded
127.0.0.1 - - [15/Jan/2024 10:30:00] "POST /webhook HTTP/1.1" 200 -
```

## Step 6: Try Other Events

The Stripe CLI can trigger many event types:

```bash
# Payment events
stripe trigger payment_intent.succeeded
stripe trigger payment_intent.payment_failed
stripe trigger charge.refunded

# Subscription events
stripe trigger customer.subscription.created
stripe trigger invoice.paid
stripe trigger invoice.payment_failed

# Customer events
stripe trigger customer.created
stripe trigger payment_method.attached
```

<!-- PRESENTER: Try a few different events -->

## Step 7: View All Available Triggers

See all available event triggers:

```bash
stripe trigger --help
```

Or list all triggers:

```bash
stripe trigger
```

In [4]:
# List available triggers
!stripe trigger 2>&1 | head -30

Checking for new versions...

A newer version of the Stripe CLI is available, please update to: v1.35.0
Trigger specific webhook events to be sent. Webhooks events created through
the trigger command will also create all necessary side-effect events that are
needed to create the triggered event as well as the corresponding API objects.

Supported events:
  account.application.deauthorized
  account.updated
  balance.available
  billing_portal.configuration.created
  billing_portal.configuration.updated
  billing_portal.session.created
  cash_balance.funds_available
  charge.captured
  charge.dispute.closed
  charge.dispute.created
  charge.dispute.updated
  charge.failed
  charge.refund.updated
  charge.refunded
  charge.succeeded
  checkout.session.async_payment_failed
  checkout.session.async_payment_succeeded
  checkout.session.completed
  checkout.session.expired
  coupon.created
  coupon.deleted
  coupon.updated


## Troubleshooting

### "Connection refused" Error

- Make sure the Flask server is running on port 4242
- Check the port matches in `stripe listen --forward-to`

### "Signature verification failed" Error

- Copy the `whsec_xxx` from `stripe listen` output
- Update `STRIPE_WEBHOOK_SECRET` in your `.env` file
- Restart the Flask server after updating `.env`

### Events Not Showing

- Verify `stripe listen` is running
- Check the Flask server logs for errors
- Ensure you're triggering events in the same Stripe account

## Bonus: Resending Events

You can resend past events from the Dashboard:

1. Go to [Developers > Webhooks](https://dashboard.stripe.com/test/webhooks)
2. Click on an endpoint
3. Find an event and click "Resend"

Or use the CLI:

```bash
stripe events resend evt_xxx
```

## Summary

In this hands-on exercise, you:

1. **Verified** Stripe CLI installation
2. **Set up** the three-terminal workflow
3. **Started** the Flask webhook server
4. **Forwarded** events using `stripe listen`
5. **Triggered** test events and verified handling

## Key Takeaways

- **stripe listen** creates a tunnel to your local server
- **stripe trigger** creates test resources and fires events
- The webhook secret from `stripe listen` is different from production
- Always test webhook handlers before deploying

## Lab Complete!

Congratulations! You've completed the Stripe Developer Tools Workshop.

### What You Learned

- **Lab 01**: Infrastructure as Code with Terraform
- **Lab 02**: Webhook integration for event handling

### Next Steps

- Explore more [Stripe CLI commands](https://stripe.com/docs/cli)
- Try [Stripe Workbench](https://dashboard.stripe.com/workbench) for API exploration
- Check out [Stripe samples](https://github.com/stripe-samples) for more examples